# Matplotlib Phase 4: Advanced Layouts & Financial Reporting
### Credit Card Risk Analysis Project

A single chart answers one question. A real risk report has to answer several at once —
which is where multi-panel layouts, color-encoded third variables, and correlation
heatmaps come in.

This notebook covers 3 topics:
10. **Subplots** — combining multiple, even different, chart types in one Figure
11. **Color Mapping (cmap)** — using a color gradient to encode a third variable
12. **Correlation Heatmaps** — visualizing feature relationships with Seaborn

**Format:** Each question has a `YOUR CODE HERE` cell to attempt first, followed by a
`Solution` cell. Try your own answer before peeking!

Run the setup cell below first — it builds a synthetic applicant pool with age, housing
status, loan purpose, income, debt, credit score, credit limit, and a simulated default
probability.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

np.random.seed(42)

n = 500

age = np.clip(np.random.normal(40, 12, n), 18, 80)

housing_status = np.random.choice(['Rent', 'Own', 'Mortgage'], size=n, p=[0.35, 0.25, 0.40])

loan_purpose = np.random.choice(
    ['Debt Consolidation', 'Credit Card', 'Home Improvement', 'Other'],
    size=n, p=[0.40, 0.30, 0.15, 0.15]
)

annual_income = np.random.lognormal(mean=10.8, sigma=0.4, size=n)
total_debt = annual_income * np.random.uniform(0.05, 0.55, size=n)
debt_to_income = (total_debt / annual_income) * 100

credit_score = np.clip(np.random.normal(680, 55, n), 300, 850)
credit_limit = np.clip(3000 + annual_income * 0.15 + np.random.normal(0, 2000, n), 500, None)

# Simulated default probability: higher DTI and lower credit score -> higher probability
raw_risk = (debt_to_income / 100) * 0.6 + ((850 - credit_score) / 550) * 0.6
default_probability = np.clip(raw_risk + np.random.normal(0, 0.08, n), 0.01, 0.95)
default = np.random.binomial(1, default_probability)

df = pd.DataFrame({
    'Age': age,
    'Housing_Status': housing_status,
    'Loan_Purpose': loan_purpose,
    'Annual_Income': annual_income,
    'Total_Debt': total_debt,
    'Debt_to_Income': debt_to_income,
    'Credit_Score': credit_score,
    'Credit_Limit': credit_limit,
    'Default_Probability': default_probability,
    'Default': default
})

print(df.shape)
df.head()


---
## Section 10: Subplots

A financial report rarely lives or dies on one chart. Subplots let you combine several
views — different chart types, different variables — into a single Figure that tells a
complete story at a glance.


**Q1.** Create a 2x2 grid of subplots with `plt.subplots(2, 2, figsize=(12, 9))`. On `axes[0, 0]`, plot a histogram of `Age`. On `axes[0, 1]`, plot a bar chart of `Housing_Status` value counts. On `axes[1, 0]`, plot a scatter of `Annual_Income` vs. `Credit_Limit`. On `axes[1, 1]`, plot a boxplot of `Debt_to_Income`. Give each subplot a short title.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].hist(df['Age'], bins=20, edgecolor='black')
axes[0, 0].set_title("Age Distribution")

housing_counts = df['Housing_Status'].value_counts()
axes[0, 1].bar(housing_counts.index, housing_counts.values)
axes[0, 1].set_title("Housing Status Counts")

axes[1, 0].scatter(df['Annual_Income'], df['Credit_Limit'], alpha=0.4)
axes[1, 0].set_title("Income vs. Credit Limit")

axes[1, 1].boxplot(df['Debt_to_Income'])
axes[1, 1].set_title("Debt-to-Income Spread")

plt.show()


**Q2.** Compute `df['Loan_Purpose'].value_counts()` and plot it as a pie chart using `ax.pie()`, passing `labels=` and `autopct='%1.1f%%'` so each slice shows its percentage.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
purpose_counts = df['Loan_Purpose'].value_counts()

fig, ax = plt.subplots()
ax.pie(purpose_counts.values, labels=purpose_counts.index, autopct='%1.1f%%')
plt.show()


**Q3.** Create a Figure with 1 row and 2 columns. On the left Axes, plot a histogram of `Age`. On the right Axes, plot the `Loan_Purpose` pie chart from Q2. This is the exact combination the phase description calls out: a histogram next to a pie chart in one Figure.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df['Age'], bins=20, edgecolor='black')
axes[0].set_title("Applicant Age Distribution")
axes[0].set_xlabel("Age")

axes[1].pie(purpose_counts.values, labels=purpose_counts.index, autopct='%1.1f%%')
axes[1].set_title("Loan Purpose Breakdown")

plt.show()


**Q4.** Build an uneven layout using `plt.subplot_mosaic()`: a mosaic string of `"AAB\nAAC"` gives you one large panel `'A'` spanning the left 2x2 area, with `'B'` and `'C'` stacked as two smaller panels on the right. In `'A'`, plot the `Annual_Income` vs. `Credit_Limit` scatter. In `'B'`, plot the `Age` histogram. In `'C'`, plot the `Housing_Status` bar chart.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
fig, axd = plt.subplot_mosaic("AAB\nAAC", figsize=(11, 7))

axd['A'].scatter(df['Annual_Income'], df['Credit_Limit'], alpha=0.4)
axd['A'].set_title("Income vs. Credit Limit")

axd['B'].hist(df['Age'], bins=15, edgecolor='black')
axd['B'].set_title("Age")

axd['C'].bar(housing_counts.index, housing_counts.values)
axd['C'].set_title("Housing Status")

plt.show()


**Q5.** Recreate the 2x2 grid from Q1 (histogram, bar, scatter, boxplot), but this time call `fig.subplots_adjust(hspace=0.4, wspace=0.3)` to add breathing room between panels so the titles and labels don't collide.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].hist(df['Age'], bins=20, edgecolor='black')
axes[0, 0].set_title("Age Distribution")

axes[0, 1].bar(housing_counts.index, housing_counts.values)
axes[0, 1].set_title("Housing Status Counts")

axes[1, 0].scatter(df['Annual_Income'], df['Credit_Limit'], alpha=0.4)
axes[1, 0].set_title("Income vs. Credit Limit")

axes[1, 1].boxplot(df['Debt_to_Income'])
axes[1, 1].set_title("Debt-to-Income Spread")

fig.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()


**Q6.** Create a Figure with 3 rows, 1 column, `sharex=True`, plotting `Annual_Income`, `Debt_to_Income`, and `Credit_Score` (as three separate histograms) stacked vertically. Loop over `axes` and the three column names together with `zip()` instead of writing each panel out by hand.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
columns_to_plot = ['Annual_Income', 'Debt_to_Income', 'Credit_Score']

fig, axes = plt.subplots(3, 1, figsize=(8, 9))

for ax, col in zip(axes, columns_to_plot):
    ax.hist(df[col], bins=25, edgecolor='black')
    ax.set_title(col)

fig.tight_layout()
plt.show()


**Q7.** Create a 2x2 grid of Axes, then flatten it with `axes.flat` (or `axes.flatten()`) and loop over it together with the four numeric columns `['Age', 'Annual_Income', 'Debt_to_Income', 'Credit_Score']`, plotting one histogram per Axes. This flattening pattern is the standard way to loop over a grid without hand-indexing `axes[0,0]`, `axes[0,1]`, etc.

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
numeric_cols = ['Age', 'Annual_Income', 'Debt_to_Income', 'Credit_Score']

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

for ax, col in zip(axes.flat, numeric_cols):
    ax.hist(df[col], bins=20, edgecolor='black')
    ax.set_title(col)

fig.tight_layout()
plt.show()


**Q8 (Capstone).** Build a 4-panel "Portfolio Overview" dashboard in a 2x2 grid: (top-left) age histogram, (top-right) loan purpose pie chart, (bottom-left) income vs. credit limit scatter, (bottom-right) debt-to-income boxplot grouped by `Housing_Status`. Give each panel a title, add one `fig.suptitle("Portfolio Overview Dashboard")`, and call `fig.tight_layout()`.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

axes[0, 0].hist(df['Age'], bins=20, edgecolor='black', color='steelblue')
axes[0, 0].set_title("Age Distribution")

axes[0, 1].pie(purpose_counts.values, labels=purpose_counts.index, autopct='%1.1f%%')
axes[0, 1].set_title("Loan Purpose Breakdown")

axes[1, 0].scatter(df['Annual_Income'], df['Credit_Limit'], alpha=0.4, color='mediumpurple')
axes[1, 0].set_title("Income vs. Credit Limit")
axes[1, 0].set_xlabel("Annual Income")
axes[1, 0].set_ylabel("Credit Limit")

dti_by_housing = [df.loc[df['Housing_Status'] == h, 'Debt_to_Income'] for h in df['Housing_Status'].unique()]
axes[1, 1].boxplot(dti_by_housing)
axes[1, 1].set_xticklabels(df['Housing_Status'].unique())
axes[1, 1].set_title("Debt-to-Income by Housing Status")

fig.suptitle("Portfolio Overview Dashboard", fontsize=15, fontweight='bold')
fig.tight_layout()
plt.show()


> **Checkpoint — Section 10:** `plt.subplots(rows, cols)` gives you a grid of Axes to fill
> in one at a time (or by looping over `.flat`), while `plt.subplot_mosaic()` gives you
> uneven, dashboard-style layouts using a simple text template. `subplots_adjust()` /
> `tight_layout()` keep panels from overlapping, and `fig.suptitle()` ties the whole grid
> together as one report.


---
## Section 11: Color Mapping (cmap)

A scatter plot already uses two dimensions (x and y). Color lets you encode a *third*
numeric variable — like coloring points from light to dark red based on default
probability — without needing another chart.


**Q9.** Create a scatter plot of `Annual_Income` (x) vs. `Total_Debt` (y), passing `c=df['Default_Probability']` and `cmap='Reds'` so points are colored by default probability. Store the return value in `scatter` and call `fig.colorbar(scatter, ax=ax, label='Default Probability')` to add a color legend.

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(df['Annual_Income'], df['Total_Debt'], c=df['Default_Probability'], cmap='Reds')
fig.colorbar(scatter, ax=ax, label='Default Probability')
ax.set_xlabel("Annual Income")
ax.set_ylabel("Total Debt")
plt.show()


**Q10.** Repeat Q9, but switch the colormap to `'viridis'` — a perceptually uniform colormap that's often preferred over `'Reds'`/`'Blues'` for general-purpose data since it doesn't imply "danger" or "safety" the way red/green do.

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(df['Annual_Income'], df['Total_Debt'], c=df['Default_Probability'], cmap='viridis')
fig.colorbar(scatter, ax=ax, label='Default Probability')
ax.set_xlabel("Annual Income")
ax.set_ylabel("Total Debt")
plt.show()


**Q11.** Repeat Q9 with `cmap='Reds'`, but explicitly fix the color scale with `vmin=0` and `vmax=1` (the true range of a probability) instead of letting Matplotlib auto-scale to the min/max of this particular sample. This matters when you want colors to be comparable across multiple separate charts.

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(df['Annual_Income'], df['Total_Debt'], c=df['Default_Probability'],
                      cmap='Reds', vmin=0, vmax=1)
fig.colorbar(scatter, ax=ax, label='Default Probability')
plt.show()


**Q12.** Color the same scatter plot by `Credit_Score` instead, using `cmap='RdYlGn'` (red-yellow-green) — an intuitive choice here since high credit score genuinely means "good".

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(df['Annual_Income'], df['Total_Debt'], c=df['Credit_Score'], cmap='RdYlGn')
fig.colorbar(scatter, ax=ax, label='Credit Score')
ax.set_xlabel("Annual Income")
ax.set_ylabel("Total Debt")
plt.show()


**Q13.** Apply color mapping to a bar chart: compute the average `Default_Probability` per `Housing_Status` group, then build a colormap manually with `mpl.cm.Reds` and `mpl.colors.Normalize(vmin=..., vmax=...)`, converting each group's average into a color via `cmap(norm(value))`, and pass that list to `ax.bar(..., color=bar_colors)`.

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
avg_risk_by_housing = df.groupby('Housing_Status')['Default_Probability'].mean()

cmap = mpl.cm.Reds
norm = mpl.colors.Normalize(vmin=avg_risk_by_housing.min(), vmax=avg_risk_by_housing.max())
bar_colors = [cmap(norm(v)) for v in avg_risk_by_housing.values]

fig, ax = plt.subplots()
ax.bar(avg_risk_by_housing.index, avg_risk_by_housing.values, color=bar_colors)
ax.set_ylabel("Average Default Probability")
ax.set_title("Average Risk by Housing Status")
plt.show()


**Q14.** Add a size dimension on top of the color dimension: repeat the `Annual_Income` vs. `Total_Debt` scatter, colored by `Default_Probability` with `cmap='Reds'`, and also set `s=df['Credit_Limit'] / 100` so marker size reflects credit limit. Now the chart encodes four variables at once (x, y, color, size).

In [ ]:
# YOUR CODE HERE


**Solution 14**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(df['Annual_Income'], df['Total_Debt'],
                      c=df['Default_Probability'], cmap='Reds',
                      s=df['Credit_Limit'] / 100, alpha=0.6)
fig.colorbar(scatter, ax=ax, label='Default Probability')
ax.set_xlabel("Annual Income")
ax.set_ylabel("Total Debt")
ax.set_title("Bubble size = Credit Limit, color = Default Probability")
plt.show()


**Q15 (Capstone).** Build the final color-mapped chart: `Annual_Income` vs. `Total_Debt`, colored by `Default_Probability` with `cmap='Reds'`, `vmin=0`/`vmax=1`, `alpha=0.7`, a colorbar labeled `"Default Probability"`, a bold title, and axis labels.

In [ ]:
# YOUR CODE HERE


**Solution 15**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(df['Annual_Income'], df['Total_Debt'],
                      c=df['Default_Probability'], cmap='Reds',
                      vmin=0, vmax=1, alpha=0.7)
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Default Probability")
ax.set_title("Income vs. Debt, Colored by Default Probability", fontsize=13, fontweight='bold')
ax.set_xlabel("Annual Income")
ax.set_ylabel("Total Debt")
plt.show()


> **Checkpoint — Section 11:** Passing a numeric array to `c=` with a `cmap=` turns a
> scatter (or, with a bit more work via `mpl.colors.Normalize`, a bar chart) into a
> chart that encodes a third variable through color. Always add a `colorbar` so the
> color scale is actually readable, and choose colormaps deliberately — sequential
> (`'Reds'`, `'viridis'`) for a single direction of "more," diverging (`'RdYlGn'`,
> `'coolwarm'`) when there's a meaningful midpoint.


---
## Section 12: Correlation Heatmaps (Seaborn)

Before feeding features into a model, you need to know how they relate to each other —
which pairs move together, and which are redundant. A correlation heatmap answers that
for every pair of numeric features at once. This is the one chart type where reaching
for Seaborn (built on top of Matplotlib) is standard practice, since `sns.heatmap()`
handles the color-matrix-plus-annotation work that would otherwise take many lines of
raw Matplotlib.


**Q16.** Select the numeric columns `['Age', 'Annual_Income', 'Total_Debt', 'Debt_to_Income', 'Credit_Score', 'Credit_Limit', 'Default_Probability']` from `df` and compute their pairwise correlation matrix with `.corr()`. Print it.

In [ ]:
# YOUR CODE HERE


**Solution 16**

In [ ]:
numeric_features = ['Age', 'Annual_Income', 'Total_Debt', 'Debt_to_Income',
                    'Credit_Score', 'Credit_Limit', 'Default_Probability']
corr_matrix = df[numeric_features].corr()
print(corr_matrix.round(2))


**Q17.** Plot `corr_matrix` as a heatmap using `sns.heatmap(corr_matrix, ax=ax)`. Even without any styling, this already shows you which features are strongly related at a glance.

In [ ]:
# YOUR CODE HERE


**Solution 17**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, ax=ax)
plt.show()


**Q18.** Repeat Q17, adding `annot=True` (print the exact correlation value in each cell) and `fmt='.2f'` (round to 2 decimal places) — turning the heatmap from a quick visual check into something precise enough to cite in a report.

In [ ]:
# YOUR CODE HERE


**Solution 18**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', ax=ax)
plt.show()


**Q19.** Repeat Q18, but set `cmap='coolwarm'` and `center=0` — this makes positive correlations red, negative correlations blue, and near-zero correlations a neutral color, which is the standard convention for correlation heatmaps.

In [ ]:
# YOUR CODE HERE


**Solution 19**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.show()


**Q20.** A correlation matrix is symmetric, so the upper and lower triangles duplicate the same information. Build a mask for the upper triangle using `np.triu(np.ones_like(corr_matrix, dtype=bool))`, then pass it to `sns.heatmap(..., mask=mask)` so only the lower triangle (plus the diagonal) is shown.

In [ ]:
# YOUR CODE HERE


**Solution 20**

In [ ]:
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.show()


**Q21.** Repeat Q20's masked lower-triangle heatmap, and add `linewidths=0.5` (thin gridlines between cells for separation) and `square=True` (force each cell to be a perfect square rather than stretched to fill the figure).

In [ ]:
# YOUR CODE HERE


**Solution 21**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True, ax=ax)
plt.show()


**Q22 (Capstone).** Which single feature correlates most strongly (by absolute value, excluding itself) with `Default_Probability`? Answer this two ways: (1) programmatically, by sorting `corr_matrix['Default_Probability'].abs()` and dropping the self-correlation; (2) visually, by producing the final polished heatmap — masked upper triangle, `annot=True`, `fmt='.2f'`, `cmap='coolwarm'`, `center=0`, `linewidths=0.5`, `square=True`, and a bold title `"Feature Correlation Matrix"`.

In [ ]:
# YOUR CODE HERE


**Solution 22**

In [ ]:
strongest = corr_matrix['Default_Probability'].drop('Default_Probability').abs().sort_values(ascending=False)
print("Strongest correlations with Default_Probability:")
print(strongest.round(3))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True, ax=ax)
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()


---
## Checkpoint: Phase 4 Complete

You've covered:
- **Subplots** — grid layouts with `plt.subplots(rows, cols)`, uneven dashboard layouts
  with `plt.subplot_mosaic()`, looping over a flattened grid with `axes.flat`, spacing
  control (`subplots_adjust`/`tight_layout`), and combining genuinely different chart
  types (histogram + pie + scatter + boxplot) into one report
- **Color mapping** — encoding a third numeric variable with `c=` and `cmap=`, adding a
  `colorbar` so it's actually readable, fixing the scale with `vmin`/`vmax`, choosing
  sequential vs. diverging colormaps deliberately, and combining color with marker size
  for a 4-variable bubble chart
- **Correlation heatmaps** — `sns.heatmap()`, `annot`/`fmt` for exact values, `cmap='coolwarm'`
  with `center=0` as the standard convention, masking the redundant upper triangle, and
  reading off which features actually matter for `Default_Probability`

This completes the full Matplotlib + Seaborn-integration toolkit for this project:
trends over time, distributions/outliers, categorical/bivariate comparisons, and now
multi-panel financial reporting.

**Next up:** this is a natural point to either apply everything to your real dataset, or
move on to a full Seaborn deep-dive (it has shortcuts for a lot of what you've been doing
by hand — e.g. `sns.pairplot()`, `sns.boxplot()` with built-in grouping). Let me know
which direction you'd like to go!
